# Model Server: Mistral-7B on Kaggle/Colab (GPU) exposed via ngrok

This notebook:
1. Loads **Mistral-7B-Instruct** locally on the notebook's free GPU (4-bit quantized so it fits).
2. Wraps it in a tiny **FastAPI** server with one endpoint, `/generate`.
3. Opens an **ngrok** tunnel so that endpoint gets a public HTTPS URL.
4. Your Streamlit Cloud app calls that URL instead of any paid AI API.

**Requirements before running:**
- On Kaggle: open this notebook, then in the right panel turn on **GPU T4 x2** (or P100) under *Accelerator*, and set *Internet* to **On**.
- Sign up for a free ngrok account at https://ngrok.com and copy your Authtoken from the dashboard.
- This notebook must **stay running** the whole time you want the Streamlit app to work - closing the tab or letting the session time out kills the URL.

In [6]:
# 1. Install dependencies
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio

In [7]:
# 2. Load Mistral-7B-Instruct in 4-bit (fits comfortably on a free T4/P100 GPU)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
# Alternative: "meta-llama/Llama-3.1-8B-Instruct" - note this one is gated on
# Hugging Face, so you'd need to accept the license there and log in with
# `from huggingface_hub import login; login("YOUR_HF_TOKEN")` first.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded.


In [8]:
# 3. generate_text(): wraps the model with Mistral's instruct chat template
def generate_text(prompt: str, max_length: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,  # gives {input_ids, attention_mask} instead of a bare tensor
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Quick sanity check
print(generate_text("Say hello in one short sentence.", max_length=30))

Hello there! I'm here to help answer any questions you might have.


In [9]:
# 4. FastAPI server - same shape as the FastAPI+ngrok snippet you already had
from fastapi import FastAPI, Request, HTTPException

API_KEY = "choose-any-secret-here"  # change this - your app.py must send the same value

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    return {
        "response": generate_text(
            data.get("prompt", ""),
            data.get("max_length", 300),
        )
    }

@app.get("/")
async def health():
    return {"status": "ok"}

In [10]:
# 5. Run the server + open the public ngrok tunnel
import threading, time
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

conf.get_default().auth_token = "3H7DJhd2wLtJsFWLdlu9IV1sl9z_RxD7VfXLp4k7EjDjL1qi"

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()
time.sleep(2)

public_url = ngrok.connect(8000)
print("Public URL for your Streamlit app's sidebar:", public_url)
print("Server key (put the same value in the Streamlit sidebar too):", API_KEY)

INFO:     Started server process [234]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Public URL for your Streamlit app's sidebar: NgrokTunnel: "https://skipper-dissuade-glue.ngrok-free.dev" -> "http://localhost:8000"
Server key (put the same value in the Streamlit sidebar too): choose-any-secret-here
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.127.33.101:0 - "POST /generate HTTP/1.

## Next step

Copy the printed `public_url` (something like `https://xxxx-xx-xx-xx.ngrok-free.app`) and paste it, along with the `API_KEY` above, into the two sidebar fields of the Streamlit app (`app.py`).

Keep this notebook running the whole time you want the app to answer requests - if the Kaggle/Colab session stops, the URL dies and the Streamlit app will show a connection error until you restart this notebook and paste the new URL.